# Prepare Quartet multiomics inputs for limma correction

Source: Quartet multiomics figshare release [10.6084/m9.figshare.22188349.v1](https://doi.org/10.6084/m9.figshare.22188349.v1) (CC BY 4.0), described in Yu et al., *Genome Biology* 24, 201 (2023), <https://doi.org/10.1186/s13059-023-03047-z>. The full Quartet matrices are already log-transformed and feature-filtered in that release. This notebook restricts the dataset to a deterministic four-client federation, aligns each omics layer to metadata, validates the resulting balanced design, and writes per-modality central matrices used downstream by `03_central_RBE.ipynb` and `04_run_fedrbe.ipynb`.

## Lab/client selection (fixed)

Lab IDs `L01..L15` in the figshare metadata are enumerated **independently within each modality**, so the same `Lxx` can refer to different institutions across modalities. Only labs that appear in **all three** modalities can host a real cross-modality client. Three labs satisfy this (`L01`, `L02`, `L05`); a fourth synthetic client merges `L03` (Metab+RNA) with `L14` (Protein) so every client covers the three layers. Within each (lab, modality), batches are pruned to a fixed total of 12 or 24 libraries per client per modality, so every modality has 6 batches × 12 libraries = 72 libraries.

| client | RNA batches | Protein batches | Metabolomics batches |
|---|---|---|---|
| `client_01_L01` | `P_ILM_L1_B1` | `ABS_QTOF6600_1` | `U_L1_01` |
| `client_02_L02` | `P_ILM_L2_B1` | `APT_QE-HFX_1` | `U_L2_01` |
| `client_03_L05_L04` | `P_ILM_L5_B1`, `R_ILM_L5_B2` | `FDU_Lumos_1`, `FDU_QE-HFX_4` | `U_L5_01`, `T_L4_01` |
| `client_04_L03_L14` | `P_BGI_L3_B1`, `R_BGI_L3_B1` | `TMO_Exploris480_1`, `TMO_QE-HFX_1` | `U_L3_01`, `U_L3_02` |

Selection rules:
- L01/L02 RNA: keep poly(A)-mRNA (`P_*`) and drop rRNA-depleted (`R_*`); polyA matches the layer measured by proteomics/metabolomics.
- L05 Proteomics: keep `FDU_Lumos_1` + `FDU_QE-HFX_4` (HFX run with the lower fraction of cells at the `log2(FOT + 0.01)` imputation floor `−6.644`); drop `FDU_QE-HFX_1`.
- L05 Metabolomics: add `T_L4_01` (alphabetical-first L04 batch) so the client has 24 metabolomics libraries, matching its RNA/Protein size. The original `lab` label `L04` is preserved in metadata.
- L03+L14: L14 is the unique Protein-only lab with two batches (24 libraries) so it pairs with L03 cleanly.

The biological covariate is the donor (`D5/D6/F7/M8`) with **`D6` as the reference level**, matching the Quartet convention where D6 is the common reference material (Yang et al., *Genome Biology* 24, 245, 2023, <https://doi.org/10.1186/s13059-023-03091-9>).


In [ ]:
library(tidyverse)
library(jsonlite)

WORKDIR <- if (dir.exists("figshare_data")) "." else "evaluation_data/multiomics"
DATA_DIR <- file.path(WORKDIR, "figshare_data")
BEFORE_DIR <- file.path(WORKDIR, "before")
dir.create(BEFORE_DIR, showWarnings = FALSE, recursive = TRUE)

# DONOR_LEVELS is ordered with the reference donor first so that R's default
# treatment contrast picks D6 as the limma intercept (matches figshare design).
DONOR_LEVELS <- c("D6", "D5", "F7", "M8")
DONOR_REFERENCE <- DONOR_LEVELS[1]
COVARIATES <- DONOR_LEVELS[-1]

## Dataset and selection registry

`SELECTED_BATCHES` lists which batches each lab contributes to its client per modality. `CLIENT_LAB_MAPS` translates each (modality, lab) pair to a client name. Together these are the only configuration needed to reproduce the four-client layout. The same constants live in `fedrbe_multiomics_utils.py` for the federated path.

In [ ]:
dataset_registry <- tribble(
  ~omics, ~datatype_raw, ~file_name, ~scale,
  "Transcriptomics", "RNA", "Transcriptomics_fulldataset_log2FPKM_r26907c180.csv", "log2(FPKM + 0.01)",
  "Proteomics", "Protein", "Proteomics_fulldataset_log2FOT_r3489c180.csv", "log2(FOT + 0.01)",
  "Metabolomics", "Metabolite", "Metabolomics_fulldataset_log2expr_r71c180.csv", "log2(expr + 1)"
)

SELECTED_BATCHES <- list(
  Transcriptomics = list(
    L01 = c("P_ILM_L1_B1"),
    L02 = c("P_ILM_L2_B1"),
    L05 = c("P_ILM_L5_B1", "R_ILM_L5_B2"),
    L03 = c("P_BGI_L3_B1", "R_BGI_L3_B1")
  ),
  Proteomics = list(
    L01 = c("ABS_QTOF6600_1"),
    L02 = c("APT_QE-HFX_1"),
    L05 = c("FDU_Lumos_1", "FDU_QE-HFX_4"),
    L14 = c("TMO_Exploris480_1", "TMO_QE-HFX_1")
  ),
  Metabolomics = list(
    L01 = c("U_L1_01"),
    L02 = c("U_L2_01"),
    L05 = c("U_L5_01"),
    L04 = c("T_L4_01"),
    L03 = c("U_L3_01", "U_L3_02")
  )
)

CLIENT_LAB_MAPS <- list(
  Transcriptomics = c(L01 = "client_01_L01", L02 = "client_02_L02",
                      L05 = "client_03_L05_L04", L03 = "client_04_L03_L14"),
  Proteomics = c(L01 = "client_01_L01", L02 = "client_02_L02",
                 L05 = "client_03_L05_L04", L14 = "client_04_L03_L14"),
  Metabolomics = c(L01 = "client_01_L01", L02 = "client_02_L02",
                   L05 = "client_03_L05_L04", L04 = "client_03_L05_L04",
                   L03 = "client_04_L03_L14")
)

CLIENT_NAMES <- c("client_01_L01", "client_02_L02",
                  "client_03_L05_L04", "client_04_L03_L14")

N_BATCHES_PER_MODALITY <- 6L
N_LIBS_PER_MODALITY <- 72L

dataset_registry

## Loading and selection helpers

The figshare metadata is loaded once for all modalities. Per-modality processing then filters to the selected (lab, batch) pairs, assigns each library to its canonical client, re-derives `batch_code` so that codes are dense-ranked within the filtered modality, and builds joint `pseudo_sample` keys of the form `{client}_{donor}_{rep}_{i}`.

Proteomics matrix columns contain periods in a few instrument tokens while metadata uses hyphens, so both sides are normalized before alignment.

In [ ]:
normalize_datatype <- function(x) {
  dplyr::recode(
    as.character(x),
    "RNA" = "Transcriptomics",
    "Protein" = "Proteomics",
    "Metabolite" = "Metabolomics",
    .default = as.character(x)
  )
}

normalize_library_ids <- function(x, omics) {
  x <- as.character(x)
  if (omics == "Proteomics") {
    stringr::str_replace_all(x, stringr::fixed("."), "-")
  } else {
    x
  }
}

load_expression_matrix <- function(path, omics) {
  raw <- readr::read_csv(path, show_col_types = FALSE)
  feature_id_col <- names(raw)[1]
  feature_cols <- feature_id_col
  feature_ids <- as.character(raw[[feature_id_col]])

  if ("metabolite_name" %in% names(raw)) {
    feature_cols <- c(feature_cols, "metabolite_name")
    feature_ids <- paste(feature_ids, raw[["metabolite_name"]], sep = "|")
  }

  expr <- raw %>%
    select(-all_of(feature_cols)) %>%
    as.data.frame(check.names = FALSE)
  rownames(expr) <- make.unique(feature_ids)
  colnames(expr) <- normalize_library_ids(colnames(expr), omics)
  expr[] <- lapply(expr, as.numeric)
  as.matrix(expr)
}

load_full_metadata <- function(path) {
  readr::read_csv(path, show_col_types = FALSE) %>%
    mutate(datatype = normalize_datatype(datatype)) %>%
    transmute(
      file = library,
      condition = sample,
      batch = batch,
      lab = lab,
      platform = platform,
      protocol = replace_na(as.character(protocol), "not_available"),
      datatype = datatype,
      rep = as.integer(rep),
      date = as.character(date)
    )
}

filter_to_selected_batches <- function(metadata, modality) {
  selection <- SELECTED_BATCHES[[modality]]
  if (is.null(selection)) stop(modality, ": no SELECTED_BATCHES entry.", call. = FALSE)
  keep_pairs <- bind_rows(lapply(names(selection), function(lab) {
    tibble(lab = lab, batch = selection[[lab]])
  }))
  filtered <- metadata %>% semi_join(keep_pairs, by = c("lab", "batch"))

  expected_n <- sum(purrr::map_int(selection, length)) * 12L  # 12 libs per batch
  if (nrow(filtered) != expected_n) {
    stop(modality, ": expected ", expected_n, " libraries after filtering, got ", nrow(filtered), call. = FALSE)
  }
  filtered
}

assign_client_and_keys <- function(metadata, modality) {
  lab_map <- CLIENT_LAB_MAPS[[modality]]
  if (is.null(lab_map)) stop(modality, ": no CLIENT_LAB_MAPS entry.", call. = FALSE)
  unmapped <- setdiff(unique(metadata$lab), names(lab_map))
  if (length(unmapped) > 0) {
    stop(modality, ": unmapped labs ", paste(unmapped, collapse = ", "), call. = FALSE)
  }

  metadata <- metadata %>%
    mutate(
      client = unname(lab_map[as.character(lab)]),
      client = factor(client, levels = CLIENT_NAMES)
    ) %>%
    mutate(batch_code = sprintf("B%02d", dense_rank(batch)))

  # pseudo_sample = {client}_{donor}_{rep}_{i}, where i indexes libraries
  # within a (client, donor, rep) cell after sorting alphabetically by batch.
  # This produces matched joint-sample IDs across all three modalities.
  metadata <- metadata %>%
    arrange(client, condition, rep, batch) %>%
    group_by(client, condition, rep) %>%
    mutate(within_cell_idx = row_number()) %>%
    ungroup() %>%
    mutate(pseudo_sample = sprintf("%s_%s_%d_%d",
                                   as.character(client),
                                   as.character(condition),
                                   rep,
                                   within_cell_idx)) %>%
    select(-within_cell_idx) %>%
    mutate(client = as.character(client))

  if (anyDuplicated(metadata$pseudo_sample) > 0) {
    stop(modality, ": duplicate pseudo_sample keys after assignment.", call. = FALSE)
  }
  metadata
}

align_expression_to_metadata <- function(expr, metadata, label) {
  missing_in_matrix <- setdiff(metadata$file, colnames(expr))
  if (length(missing_in_matrix) > 0) {
    stop(label, ": metadata samples missing from expression matrix: ",
         paste(head(missing_in_matrix, 10), collapse = ", "), call. = FALSE)
  }
  metadata <- metadata %>% arrange(client, batch, condition, rep)
  list(expr = expr[, metadata$file, drop = FALSE], metadata = metadata)
}

validate_filtered_design <- function(metadata, label) {
  if (nrow(metadata) != N_LIBS_PER_MODALITY) {
    stop(label, ": expected ", N_LIBS_PER_MODALITY, " samples, got ", nrow(metadata), call. = FALSE)
  }
  if (n_distinct(metadata$batch) != N_BATCHES_PER_MODALITY) {
    stop(label, ": expected ", N_BATCHES_PER_MODALITY, " batches.", call. = FALSE)
  }
  if (!setequal(unique(metadata$condition), DONOR_LEVELS)) {
    stop(label, ": expected donor levels D5/D6/F7/M8.", call. = FALSE)
  }

  donor_counts <- metadata %>% count(condition)
  expected_per_donor <- N_LIBS_PER_MODALITY %/% length(DONOR_LEVELS)
  if (any(donor_counts$n != expected_per_donor)) {
    stop(label, ": expected ", expected_per_donor, " samples per donor.", call. = FALSE)
  }

  batch_donor_counts <- metadata %>% count(batch, condition)
  bad <- batch_donor_counts %>% filter(n != 3)
  expected_cells <- N_BATCHES_PER_MODALITY * length(DONOR_LEVELS)
  if (nrow(bad) > 0 || nrow(batch_donor_counts) != expected_cells) {
    stop(label, ": expected exactly 3 replicates per donor in every batch.", call. = FALSE)
  }

  client_counts <- metadata %>% count(client)
  if (!setequal(client_counts$client, CLIENT_NAMES)) {
    stop(label, ": expected exactly the four canonical clients.", call. = FALSE)
  }
  if (any(!client_counts$n %in% c(12L, 24L))) {
    stop(label, ": each client must have 12 or 24 libraries.", call. = FALSE)
  }

  TRUE
}

write_feature_matrix <- function(expr, path) {
  out <- as.data.frame(expr, check.names = FALSE) %>% rownames_to_column("rowname")
  write.table(out, file = path, sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)
}

## Write per-modality central inputs

For each modality the per-modality central matrix and metadata are written to `before/<modality>/`. The metadata carries the deterministic four-client mapping, batch_code, and joint `pseudo_sample` keys. FedRBE client folders themselves are produced by `prepare_fedrbe_clients` in `fedrbe_multiomics_utils.py` from the same metadata.

In [ ]:
full_metadata <- load_full_metadata(file.path(DATA_DIR, "meta_full_dataset_3omics.csv"))

prep_summaries <- purrr::map_dfr(seq_len(nrow(dataset_registry)), function(i) {
  selected <- dataset_registry[i, ]
  omics <- selected$omics
  message("Preparing ", omics)

  expr <- load_expression_matrix(file.path(DATA_DIR, selected$file_name), omics)

  metadata <- full_metadata %>%
    filter(datatype == omics) %>%
    mutate(
      file = normalize_library_ids(file, omics),
      condition = factor(condition, levels = DONOR_LEVELS)
    ) %>%
    filter_to_selected_batches(omics) %>%
    mutate(condition = as.character(condition)) %>%
    assign_client_and_keys(omics)

  validate_filtered_design(metadata, omics)
  aligned <- align_expression_to_metadata(expr, metadata, omics)
  expr <- aligned$expr
  metadata <- aligned$metadata

  variances <- apply(expr, 1, var, na.rm = TRUE)
  stats <- tibble(
    omics = omics,
    features = nrow(expr),
    samples = ncol(expr),
    batches = n_distinct(metadata$batch),
    clients = n_distinct(metadata$client),
    donors = n_distinct(metadata$condition),
    missing_cells = sum(is.na(expr)),
    rows_with_any_na = sum(rowSums(is.na(expr)) > 0),
    zero_or_na_variance_rows = sum(!is.finite(variances) | variances == 0)
  )

  if (stats$missing_cells > 0) stop(omics, ": source matrix contains missing values.", call. = FALSE)
  if (stats$zero_or_na_variance_rows > 0) stop(omics, ": source matrix contains zero-variance rows.", call. = FALSE)

  modality_dir <- file.path(BEFORE_DIR, omics)
  dir.create(modality_dir, showWarnings = FALSE, recursive = TRUE)

  write_feature_matrix(expr, file.path(modality_dir, "central_intensities_log_UNION.tsv"))
  write.table(metadata, file = file.path(modality_dir, "metadata.tsv"),
              sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

  stats
})

prep_summaries

## Save preparation summary

The summary records the matrix shape and basic validation checks used by the central correction notebook.

In [ ]:
write.table(prep_summaries, file = file.path(BEFORE_DIR, "preparation_summary.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

datainfo <- list(
  source = "Quartet full multiomics Figshare 10.6084/m9.figshare.22188349.v1",
  sample_set = "four-client federation, 72 libraries per modality",
  correction_batch = "batch",
  reference_donor = DONOR_REFERENCE,
  preserved_biology = DONOR_LEVELS,
  client_names = CLIENT_NAMES,
  modalities = prep_summaries
)
writeLines(jsonlite::toJSON(datainfo, pretty = TRUE, auto_unbox = TRUE), file.path(BEFORE_DIR, "datainfo.json"))

cat("Prepared inputs written to", normalizePath(BEFORE_DIR), "\n")

## Session information

In [ ]:
sessionInfo()